# Crime Data Recommendation System

## Data Processing

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Load data
csv_path = Path("data/chicago_crimes.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
else:
    url = "https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=1000000"
    df = pd.read_csv(url)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)

# Only include the following crimes
crime_types = [
    "ROBBERY",
    "THEFT",
    "MOTOR VEHICLE THEFT",
    "BURGLARY",
    "CRIMINAL DAMAGE",
    "CRIMINAL TRESPASS",
]
df = df.loc[df["primary_type"].isin(crime_types)].copy()
print(f"Matching crime rows: {len(df):,}")

# Drop un-needed features

## EDA - Exploratory Data Analysis

In [ ]:
# Distribution of crime types
crime_counts = df["primary_type"].value_counts().sort_values()
ax = crime_counts.plot.barh(figsize=(10, 4), color="steelblue")
ax.set(title="Reports by crime type", xlabel="Number of reports", ylabel="Crime type")
ax.bar_label(ax.containers[0], padding=3)
ax.margins(x=0.15)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of locations
top_locations = df["location_description"].value_counts().head(10).sort_values()
ax = top_locations.plot.barh(figsize=(11, 5), color="teal")
ax.set(title="Top 10 location types", xlabel="Number of reports", ylabel="Location type")
ax.bar_label(ax.containers[0], padding=3)
ax.margins(x=0.15)
plt.tight_layout()
plt.show()

In [ ]:
arrest_counts = df["arrest"].value_counts().reindex([True, False], fill_value=0)
arrest_counts.index = ["Arrest", "No arrest"]
ax = arrest_counts.plot.bar(figsize=(6, 4), color=["teal", "steelblue"], rot=0)
ax.set(title="Arrests vs. no arrests", xlabel="Arrest outcome", ylabel="Number of reports")
ax.bar_label(ax.containers[0], padding=3)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()

In [ ]:
crime_dates = pd.to_datetime(df["date"], errors="coerce")
months = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]
monthly_counts = crime_dates.dt.month_name().value_counts().reindex(months, fill_value=0)

ax = monthly_counts.plot.bar(figsize=(11, 4), color="steelblue", rot=45)
ax.set(title="Reports by month (all years combined)", xlabel="Month", ylabel="Number of reports")
ax.bar_label(ax.containers[0], padding=3)
ax.margins(y=0.15)
plt.tight_layout()
plt.show()


weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_counts = crime_dates.dt.day_name().value_counts().reindex(weekdays, fill_value=0)
ax = weekday_counts.plot.bar(figsize=(9, 4), color="teal", rot=45)
ax.set(title="Reports by day of the week", xlabel="Day of the week", ylabel="Number of reports")
plt.tight_layout()
plt.show()


hourly_counts = crime_dates.dt.hour.value_counts().reindex(range(24), fill_value=0)
ax = hourly_counts.plot.bar(figsize=(10, 4), color="steelblue", rot=0)
ax.set(title="Reports by hour of the day", xlabel="Hour (0 = midnight)", ylabel="Number of reports")
plt.tight_layout()
plt.show()

# Modeling

### Model 1 - Arrest Made (Logistic Regression)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

# Use the crime type and location as inputs.
dates = pd.to_datetime(df["date"])
features = df[["primary_type", "location_description", "community_area"]].astype(str)

# Train on the older 80% of reports and test on the newer 20%.
split_date = dates.quantile(0.8)
training_rows = dates < split_date
testing_rows = ~training_rows

# Convert category names into numeric input columns.
encoder = OneHotEncoder(handle_unknown="ignore")
X_train = encoder.fit_transform(features.loc[training_rows])
X_test = encoder.transform(features.loc[testing_rows])

# Use 1 for an arrest and 0 for no arrest.
y_train = df.loc[training_rows, "arrest"].astype(int).to_numpy()
y_test = df.loc[testing_rows, "arrest"].astype(int).to_numpy()

# Fit the model and predict arrests for the test reports.
arrest_model = LogisticRegression(max_iter=1000)
arrest_model.fit(X_train, y_train)
arrest_predictions = arrest_model.predict(X_test)

# Randomly guess "arrest" at the rate seen in the training data.
arrest_rate = y_train.mean()
rng = np.random.default_rng(42)
random_predictions = rng.random(len(y_test)) < arrest_rate

# Compare the percentage of correct predictions.
accuracy = pd.Series({
    "Logistic regression": (arrest_predictions == y_test).mean() * 100,
    "Random guess": (random_predictions == y_test).mean() * 100,
})

ax = accuracy.plot.bar(figsize=(7, 4), rot=0, color=["steelblue", "darkorange"])
ax.set(title="Arrest prediction accuracy", ylabel="Correct predictions (%)", ylim=(0, 105))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.show()

### Model 2 - Location 

### Model 3 - Temporal Model

### Model 4 - Patrol Recommendation (Hybrid Ranking)

Giga model